# 🚗 Car Finder (model years 2000–2026)

Type the car you want in plain English and the notebook tells you **whether it is for sale**, then shows the matching cars.

**How to use it**
1. Click **Runtime → Run all** (or press the ▶ button on each cell from top to bottom).
2. Scroll to the search box at the bottom, type something like `white Toyota Corolla 2018`, and press **Search**.

Nothing needs to be installed: it uses `pandas` and `ipywidgets`, which Colab already has.

**What happens inside**
| Step | What it does |
|---|---|
| 1. Dataset | Creates a table of about 1,700 cars (or loads your own CSV). |
| 2. Query reader | Turns your sentence into conditions: make, model, colour, year, price, mileage, fuel, gearbox, city. |
| 3. Search | Filters the table with pandas and keeps only cars that are **For Sale**. |
| 4. Answer | Shows **Yes** or **No**, the matching cars, and the closest alternatives when nothing matches. |

## 1. Setup and dataset

In [ ]:
import os, random, re, html
from dataclasses import dataclass, field
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

In [ ]:
def make_dataset():
    """Builds the synthetic car listings (same data every time because the random seed is fixed)."""
    random.seed(42)
    SCALE = 1
    Y0, Y1 = 2000, 2026

    # make -> [(model, body, new_price_usd, engine_cc, fuel_code, first_year, last_year)]
    # fuel_code: P petrol, D diesel, PD petrol/diesel, PH petrol or hybrid (hybrid only from 2010), E electric only
    CATALOG = {
        "Toyota": [("Corolla", "Sedan", 24000, 1800, "PH", 2000, 2026), ("Camry", "Sedan", 29000, 2500, "PH", 2000, 2026),
                   ("Yaris", "Hatchback", 18000, 1500, "P", 2000, 2026), ("Land Cruiser", "SUV", 75000, 4000, "PD", 2000, 2026),
                   ("Hilux", "Pickup", 35000, 2800, "D", 2000, 2026), ("Prius", "Hatchback", 28000, 1800, "H", 2003, 2026)],
        "Honda": [("Civic", "Sedan", 25000, 1800, "PH", 2000, 2026), ("City", "Sedan", 18000, 1500, "P", 2000, 2026),
                  ("Accord", "Sedan", 30000, 2000, "PH", 2000, 2026), ("CR-V", "SUV", 33000, 1500, "PH", 2000, 2026),
                  ("HR-V", "SUV", 26000, 1500, "P", 2015, 2026)],
        "Suzuki": [("Alto", "Hatchback", 8000, 660, "P", 2000, 2026), ("Cultus", "Hatchback", 11000, 1000, "P", 2000, 2017),
                   ("Swift", "Hatchback", 15000, 1200, "P", 2005, 2026), ("Wagon R", "Hatchback", 10000, 1000, "P", 2000, 2026),
                   ("Vitara", "SUV", 24000, 1600, "P", 2000, 2026)],
        "Hyundai": [("Elantra", "Sedan", 22000, 2000, "P", 2000, 2026), ("Tucson", "SUV", 30000, 2000, "PD", 2004, 2026),
                    ("Sonata", "Sedan", 27000, 2500, "PH", 2000, 2026), ("Ioniq 5", "SUV", 45000, 0, "E", 2021, 2026)],
        "Kia": [("Picanto", "Hatchback", 12000, 1000, "P", 2004, 2026), ("Sportage", "SUV", 29000, 2000, "PD", 2000, 2026),
                ("Sorento", "SUV", 35000, 2400, "PD", 2002, 2026), ("EV6", "SUV", 48000, 0, "E", 2021, 2026)],
        "Nissan": [("Sunny", "Sedan", 14000, 1500, "P", 2000, 2022), ("Patrol", "SUV", 60000, 4000, "PD", 2000, 2026),
                   ("Skyline", "Coupe", 38000, 3000, "P", 2000, 2019), ("Leaf", "Hatchback", 30000, 0, "E", 2011, 2026)],
        "Ford": [("Focus", "Hatchback", 20000, 1600, "P", 2000, 2018), ("Mustang", "Coupe", 40000, 5000, "P", 2000, 2026),
                 ("Ranger", "Pickup", 36000, 2200, "D", 2000, 2026), ("F-150", "Pickup", 45000, 3500, "P", 2000, 2026),
                 ("Explorer", "SUV", 42000, 3500, "P", 2000, 2026)],
        "BMW": [("3 Series", "Sedan", 45000, 2000, "PD", 2000, 2026), ("5 Series", "Sedan", 60000, 2000, "PD", 2000, 2026),
                ("X5", "SUV", 70000, 3000, "PD", 2000, 2026), ("Z4", "Convertible", 52000, 2000, "P", 2003, 2026)],
        "Mercedes-Benz": [("C-Class", "Sedan", 48000, 2000, "PD", 2000, 2026), ("E-Class", "Sedan", 62000, 2000, "PD", 2000, 2026),
                          ("S-Class", "Sedan", 115000, 3000, "PD", 2000, 2026), ("GLC", "SUV", 55000, 2000, "PD", 2015, 2026)],
        "Volkswagen": [("Golf", "Hatchback", 26000, 1400, "PD", 2000, 2026), ("Passat", "Sedan", 30000, 1800, "PD", 2000, 2022),
                       ("Tiguan", "SUV", 32000, 1400, "PD", 2008, 2026), ("Beetle", "Hatchback", 22000, 2000, "P", 2000, 2019)],
        "Mazda": [("Mazda 3", "Sedan", 24000, 2000, "P", 2003, 2026), ("CX-5", "SUV", 30000, 2500, "P", 2012, 2026),
                  ("MX-5", "Convertible", 31000, 2000, "P", 2000, 2026)],
        "Chevrolet": [("Camaro", "Coupe", 38000, 3600, "P", 2000, 2024), ("Corvette", "Coupe", 68000, 6200, "P", 2000, 2026),
                      ("Silverado", "Pickup", 45000, 5300, "P", 2000, 2026)],
        "Tesla": [("Model S", "Sedan", 95000, 0, "E", 2012, 2026), ("Model 3", "Sedan", 42000, 0, "E", 2017, 2026),
                  ("Model Y", "SUV", 50000, 0, "E", 2020, 2026), ("Model X", "SUV", 100000, 0, "E", 2016, 2026)],
    }
    COLORS = ["White", "Black", "Silver", "Red", "Blue", "Green", "Grey", "Yellow", "Maroon", "Gold"]
    COLOR_W = [20, 17, 17, 9, 11, 5, 12, 2, 4, 3]
    CITIES = ["Lahore", "Karachi", "Islamabad", "Rawalpindi", "Faisalabad", "Multan", "Peshawar"]


    def pick_fuel(code, year):
        if code == "E": return "Electric"
        if code == "H": return "Hybrid"
        if code == "PH": return "Hybrid" if year >= 2010 and random.random() < 0.3 else "Petrol"
        if code == "PD": return random.choice(["Petrol", "Diesel"])
        return "Diesel" if code == "D" else "Petrol"


    rows = []
    for make, models in CATALOG.items():
        for model, body, base, cc, code, first, last in models:
            lo, hi = max(first, Y0), min(last, Y1)
            for _ in range(random.randint(22, 38) * SCALE):
                year = random.randint(lo, hi)
                age = Y1 + 1 - year
                mileage = max(300, round(random.gauss(14000 * (age - 0.5), 8000), -2))
                price = base * (0.88 ** (age - 1)) * (1 - mileage / 1_500_000) * random.uniform(0.9, 1.1)
                price = max(price, base * 0.08)
                fuel = pick_fuel(code, year)
                rows.append({
                    "make": make, "model": model, "year": year,
                    "color": random.choices(COLORS, COLOR_W)[0],
                    "body_type": body, "fuel": fuel,
                    "transmission": "Automatic" if fuel == "Electric" else
                                    random.choices(["Manual", "Automatic"], [35, 65] if year > 2012 else [55, 45])[0],
                    "engine_cc": cc if fuel != "Electric" else 0,
                    "mileage_km": int(mileage), "price_usd": int(round(price, -1)),
                    "city": random.choice(CITIES),
                    "status": random.choices(["For Sale", "Sold", "Reserved"], [62, 30, 8])[0],
                })

    df = pd.DataFrame(rows).sample(frac=1, random_state=1).reset_index(drop=True)
    df.insert(0, "car_id", [f"CAR-{i:04d}" for i in range(1, len(df) + 1)])
    return df


# If a file named cars_2000_2026.csv has been uploaded to Colab (folder icon on the left), it is used instead.
# It must have the columns: car_id, make, model, year, color, body_type, fuel, transmission,
# engine_cc, mileage_km, price_usd, city, status   (status = For Sale / Sold / Reserved)
CSV = "cars_2000_2026.csv"
if os.path.exists(CSV):
    df = pd.read_csv(CSV)
    print(f"Loaded your file {CSV}")
else:
    df = make_dataset()
    df.to_csv(CSV, index=False)
    print(f"Created a new dataset and saved it as {CSV}")

for col in ["make", "model", "color", "body_type", "fuel", "transmission", "city", "status"]:
    df[col] = df[col].astype(str).str.strip()

print(f"{len(df):,} cars | model years {df.year.min()}–{df.year.max()} | {(df.status == 'For Sale').sum():,} for sale | {df.make.nunique()} makes")
df.head(8)

### A quick look at the data
The table has one row per car. `status` tells us whether the car is still available.

In [ ]:
summary = pd.DataFrame({
    "For Sale": df[df.status == "For Sale"].groupby("make").size(),
    "Sold": df[df.status == "Sold"].groupby("make").size(),
    "Reserved": df[df.status == "Reserved"].groupby("make").size(),
}).fillna(0).astype(int)
summary["Total"] = summary.sum(axis=1)
summary.sort_values("Total", ascending=False)

## 2. The search engine
**Reading the sentence.** The reader learns its words from the dataset (every make, model, colour, city…), then looks for them in your sentence together with numbers such as years (`2018`, `2010-2015`, `2015 or newer`), prices (`under 8k`, `between 5000 and 9000`) and mileage (`under 100000 km`).
Any word it cannot place (for example a brand that is not in the data) is reported as *not in our listings* instead of being silently ignored.

**Searching.** All conditions must be true together. The answer is **Yes** only if at least one matching car has the status *For Sale*.

In [ ]:
from __future__ import annotations
@dataclass
class Criteria:
    makes: list[str] = field(default_factory=list)
    models: list[str] = field(default_factory=list)
    colors: list[str] = field(default_factory=list)
    body_types: list[str] = field(default_factory=list)
    fuels: list[str] = field(default_factory=list)
    transmissions: list[str] = field(default_factory=list)
    cities: list[str] = field(default_factory=list)
    year_min: int | None = None
    year_max: int | None = None
    price_min: float | None = None
    price_max: float | None = None
    mileage_max: float | None = None
    unknown: list[str] = field(default_factory=list)   # words that match nothing in the dataset

    def active(self) -> dict:
        """Only the filters actually in use, as {label: value}."""
        out = {}
        for label, val in [("make", self.makes), ("model", self.models), ("color", self.colors),
                           ("body type", self.body_types), ("fuel", self.fuels),
                           ("transmission", self.transmissions), ("city", self.cities)]:
            if val:
                out[label] = " / ".join(val)
        if self.year_min is not None or self.year_max is not None:
            lo, hi = self.year_min, self.year_max
            out["year"] = str(lo) if lo == hi else f"{lo or '…'}–{hi or '…'}"
        if self.price_min is not None or self.price_max is not None:
            out["price"] = f"${self.price_min or 0:,.0f} – " + (f"${self.price_max:,.0f}" if self.price_max else "any")
        if self.mileage_max is not None:
            out["mileage"] = f"≤ {self.mileage_max:,.0f} km"
        if self.unknown:
            out["not in our listings"] = ", ".join(self.unknown)
        return out

    def without(self, label: str) -> "Criteria":
        c = Criteria(**self.__dict__)
        mapping = {"make": "makes", "model": "models", "color": "colors", "body type": "body_types",
                   "fuel": "fuels", "transmission": "transmissions", "city": "cities"}
        if label in mapping:
            setattr(c, mapping[label], [])
        elif label == "year":
            c.year_min = c.year_max = None
        elif label == "price":
            c.price_min = c.price_max = None
        elif label == "mileage":
            c.mileage_max = None
        elif label == "not in our listings":
            c.unknown = []
        return c


# ---------- parsing ----------
def _price_ctx(y: str, t: str) -> bool:
    """True if a 4-digit number like 2000 is clearly a price, not a model year."""
    return bool(
        re.search(r"(?:under|below|less than|cheaper than|max(?:imum)?|upto|up to|within|budget(?: of)?|over|above|"
                  r"more than|at least|min(?:imum)?|between)\s*\$?\s*" + y + r"(?!\d)", t)
        or re.search(r"\$\s?" + y, t) or re.search(y + r"\s?(?:usd|dollars)", t)
        or re.search(y + r"\s*(?:and|to|-)\s*\$?\s*\d{4,}", t))


def _flex(v: str) -> str:
    return re.escape(v.lower()).replace(r"\ ", r"[\s-]?").replace(r"\-", r"[\s-]?")


def _find(text: str, vocab) -> list[str]:
    found = []
    for v in sorted(set(vocab), key=len, reverse=True):          # longest first: "Land Cruiser" before "Land"
        pat = r"(?<![a-z0-9])" + _flex(v) + r"s?(?![a-z0-9])"
        if re.search(pat, text):
            found.append(v)
    return found


def _num(s: str) -> float:
    s = s.lower().replace(",", "").replace("$", "").strip()
    return float(s[:-1]) * 1000 if s.endswith("k") else float(s)


ALIASES = {"vw": "Volkswagen", "ev": "Electric", "evs": "Electric", "electrics": "Electric", "hybrids": "Hybrid",
           "benzin": "Petrol", "gasoline": "Petrol",
           "chevy": "Chevrolet", "landcruiser": "Land Cruiser", "gray": "Grey", "truck": "Pickup", "pickups": "Pickup"}
STOP = set("""a an the is are was be any for sale selling available availability car cars vehicle vehicles in on at with
under below over above than less more between and or to from after before since until till i we you my me want wanted
looking look need needed do does have has show find get buy new used old cheap cheaper best good nice please can could
km kms miles mile mi kilometers kilometres usd dollars dollar model year color colour of least max maximum min minimum
upto up budget within auto automatic manual stick transmission engine petrol diesel gas fuel city price mileage there
that this which it its one some around about approximately approx under ready available stock pickup suv sedan coupe hatchback
convertible wagon electric hybrid ev newer older later earlier onwards onward what whats where how much many sold""".split())

NUM = r"\$?\s?(\d[\d,]*(?:\.\d+)?\s?k?)"


def parse_query(text: str, vocab: dict) -> Criteria:
    t = " " + text.lower().strip() + " "
    t = re.sub(r"(?<![a-z])(?:mercedes[\s-]?benz|mercedes|benz|merc)(?![a-z])", "mercedes-benz", t)
    for alias, real in ALIASES.items():          # normalise nicknames before matching
        t = re.sub(r"(?<![a-z])" + alias + r"(?![a-z])", real.lower(), t)
    c = Criteria()
    c.models = _find(t, vocab["model"])
    c.makes = _find(t, vocab["make"])
    # a model already implies its make
    if c.models:
        implied = {mk for m in c.models for mk in vocab["model_makes"].get(m, ())}
        c.makes = [m for m in c.makes if m in implied]
    c.colors = _find(t, vocab["color"])
    c.body_types = _find(t, vocab["body_type"])
    c.fuels = _find(t, vocab["fuel"])
    c.cities = _find(t, vocab["city"])
    if re.search(r"\bauto(matic)?\b", t):
        c.transmissions = ["Automatic"]
    elif re.search(r"\bmanual\b|\bstick\b", t):
        c.transmissions = ["Manual"]

    # mileage first, so those numbers are not mistaken for price/year
    m = re.search(r"(?:under|below|less than|max|within|upto|up to)?\s*" + NUM + r"\s?(?:km|kms|kilometers|kilometres|miles|mi)\b", t)
    if m:
        c.mileage_max = _num(m.group(1)) * (1.609 if re.search(r"miles|mi\b", m.group(0)) else 1)
        t = t.replace(m.group(0), " ")

    # year range / single year (model years are 4-digit tokens from 2000 to 2026)
    years_rng = re.search(r"\b(20(?:0\d|1\d|2[0-6]))\s?(?:-|–|to|and|till|until)\s?(20(?:0\d|1\d|2[0-6]))\b", t)
    if years_rng:
        a, b = sorted((int(years_rng.group(1)), int(years_rng.group(2))))
        c.year_min, c.year_max = a, b
        t = t.replace(years_rng.group(0), " ")
    else:
        single = [y for y in re.findall(r"\b(20(?:0\d|1\d|2[0-6]))\b", t) if not _price_ctx(y, t)]
        if len(single) == 1:
            y = int(single[0])
            if re.search(r"(?:after|newer than|since|from)\s+" + single[0], t) or \
               re.search(single[0] + r"\s*(?:or|and)?\s*(?:newer|later|above|onwards?|\+)", t):
                c.year_min = y
            elif re.search(r"(?:before|older than|until|till)\s+" + single[0], t) or \
               re.search(single[0] + r"\s*(?:or|and)?\s*(?:older|earlier|below)", t):
                c.year_max = y
            else:
                c.year_min = c.year_max = y
            t = t.replace(single[0], " ", 1)
        elif len(single) >= 2:
            ys = sorted(int(x) for x in single)
            c.year_min, c.year_max = ys[0], ys[-1]
            for x in single:
                t = t.replace(x, " ", 1)

    # price
    pr = re.search(r"(?:between)\s*" + NUM + r"\s*(?:and|to|-)\s*" + NUM, t) or \
         re.search(NUM + r"\s*(?:-|to)\s*" + NUM, t)
    if pr and 'between' in pr.group(0) or (pr and "$" in pr.group(0)):
        lo, hi = sorted((_num(pr.group(1)), _num(pr.group(2))))
        c.price_min, c.price_max = lo, hi
    else:
        mx = re.search(r"(?:under|below|less than|cheaper than|max(?:imum)?|upto|up to|within|budget(?: of)?)\s*" + NUM, t)
        mn = re.search(r"(?:over|above|more than|at least|min(?:imum)?)\s*" + NUM, t)
        if mx:
            c.price_max = _num(mx.group(1))
        if mn:
            c.price_min = _num(mn.group(1))
        if not mx and not mn:
            d = re.search(r"\$\s?(\d[\d,]*k?)|(\d[\d,]*k?)\s?(?:usd|dollars)", t)
            if d:
                c.price_max = _num(d.group(1) or d.group(2))

    # anything left that we cannot place (e.g. a brand we do not list) must not be silently ignored
    left = t
    for v in (vocab["make"] + vocab["model"] + vocab["color"] + vocab["body_type"] + vocab["city"]):
        left = re.sub(r"(?<![a-z0-9])" + _flex(v) + r"s?(?![a-z0-9])", " ", left)
    words = re.findall(r"[a-z][a-z\-]{2,}", left)
    c.unknown = [w.title() for w in dict.fromkeys(words) if w not in STOP]
    return c

In [ ]:
VOCAB_COLS = ["make", "model", "color", "body_type", "fuel", "transmission", "city"]
FOR_SALE = "For Sale"

# the words the reader can recognise, learned from the data
VOCAB = {c: sorted(df[c].unique()) for c in VOCAB_COLS}
VOCAB["model_makes"] = df.groupby("model")["make"].agg(set).to_dict()


def apply_criteria(frame, c):
    if c.unknown:                                  # a word we have never heard of cannot match anything
        return frame.iloc[0:0]
    keep = pd.Series(True, index=frame.index)
    for col, vals in [("make", c.makes), ("model", c.models), ("color", c.colors), ("body_type", c.body_types),
                      ("fuel", c.fuels), ("transmission", c.transmissions), ("city", c.cities)]:
        if vals:
            keep &= frame[col].isin(vals)
    if c.year_min is not None: keep &= frame["year"] >= c.year_min
    if c.year_max is not None: keep &= frame["year"] <= c.year_max
    if c.price_min is not None: keep &= frame["price_usd"] >= c.price_min
    if c.price_max is not None: keep &= frame["price_usd"] <= c.price_max
    if c.mileage_max is not None: keep &= frame["mileage_km"] <= c.mileage_max
    return frame[keep]


@dataclass
class Result:
    criteria: Criteria
    available: pd.DataFrame        # matching cars that are for sale, cheapest first
    unavailable: pd.DataFrame      # matching cars that are sold or reserved
    alternatives: list             # (condition we relaxed, for-sale cars without it)

    @property
    def found(self):
        return len(self.available) > 0


def search(frame, c):
    matches = apply_criteria(frame, c)
    available = matches[matches["status"] == FOR_SALE].sort_values("price_usd")
    unavailable = matches[matches["status"] != FOR_SALE]
    alternatives = []
    if available.empty:                            # nothing found: try dropping one condition at a time
        for label in c.active():
            alt = apply_criteria(frame, c.without(label))
            alt = alt[alt["status"] == FOR_SALE]
            if not alt.empty:
                alternatives.append((label, alt.sort_values("price_usd")))
        alternatives.sort(key=lambda x: len(x[1]))  # smallest change first
    return Result(c, available, unavailable, alternatives)

## 3. The search box
Run the cell below, then type your request and press **Search** (or click one of the examples).

In [ ]:
COLS = ["car_id", "make", "model", "year", "color", "body_type", "fuel", "transmission", "mileage_km", "price_usd", "city"]
LABELS = {"car_id": "ID", "make": "Make", "model": "Model", "year": "Year", "color": "Colour", "body_type": "Body",
          "fuel": "Fuel", "transmission": "Gearbox", "mileage_km": "Mileage (km)", "price_usd": "Price (USD)",
          "city": "City", "status": "Status"}
ROW_LIMIT = 500      # most rows shown at once
EXAMPLES = ["white Toyota Corolla 2018", "electric Tesla under 40k", "automatic Honda Civic 2015 or newer",
            "diesel pickup under 30k", "blue Ferrari 2010"]

text = widgets.Text(placeholder="e.g. silver Suzuki Swift 2019 in Lahore", layout=widgets.Layout(width="68%"),
                    continuous_update=False)
button = widgets.Button(description="Search", button_style="primary", icon="search", layout=widgets.Layout(width="130px"))
out = widgets.Output()


def banner(message, ok):
    bg, fg, bar = ("#e8f5ee", "#14532d", "#16a34a") if ok else ("#fdecec", "#7f1d1d", "#dc2626")
    display(HTML(f'<div style="background:{bg};color:{fg};border-left:6px solid {bar};padding:12px 16px;'
                 f'border-radius:8px;font-size:17px;font-weight:600;margin:10px 0">{message}</div>'))


def chips(active):
    parts = []
    for k, v in active.items():
        bad = k == "not in our listings"
        bg, fg = ("#fee2e2", "#991b1b") if bad else ("#e2e8f0", "#1e293b")
        parts.append(f'<span style="display:inline-block;background:{bg};color:{fg};padding:2px 10px;margin:0 6px 6px 0;'
                     f'border-radius:999px;font-size:13px">{html.escape(k)}: {html.escape(str(v))}</span>')
    display(HTML("".join(parts)))


def show_table(frame, extra=(), limit=ROW_LIMIT):
    t = frame[COLS + list(extra)].head(limit).rename(columns=LABELS)
    display(t.style.hide(axis="index").format({"Mileage (km)": "{:,}", "Price (USD)": "{:,}"}))


def run_search(query):
    with out:
        clear_output(wait=True)
        query = query.strip()
        if not query:
            return
        c = parse_query(query, VOCAB)
        active = c.active()
        if not active:
            display(HTML("<p>Add a make, model, colour, year or price so we can search. Example: <i>white Honda Accord 2012</i>.</p>"))
            return
        r = search(df, c)
        chips(active)
        if r.found:
            n = len(r.available)
            banner(f"Yes, {n:,} matching car{'s' if n != 1 else ''} for sale.", True)
            if n > ROW_LIMIT:
                display(HTML(f"<p>Showing the {ROW_LIMIT} cheapest of {n:,} matches.</p>"))
            show_table(r.available)
            dl = widgets.Button(description="Download results (CSV)", icon="download")

            def download(_, rows=r.available[COLS]):
                with out:                              # Colab needs the output area to start the download
                    try:
                        from google.colab import files
                        rows.to_csv("results.csv", index=False)
                        files.download("results.csv")
                    except ImportError:
                        print("The download button works inside Google Colab.")
            dl.on_click(download)
            display(dl)
            return
        banner("No, nothing matching is for sale right now.", False)
        if c.unknown:
            display(HTML(f"<p>We have no listings for <b>{html.escape(', '.join(c.unknown))}</b>. "
                         "Check the spelling, or ask for a different make or model.</p>"))
        if len(r.unavailable):
            display(HTML(f"<p>{len(r.unavailable)} matching car(s) exist but are already sold or reserved:</p>"))
            show_table(r.unavailable, extra=["status"], limit=50)
        if r.alternatives:
            display(HTML("<h3>Closest alternatives</h3>"))
            for label, alt in r.alternatives[:3]:
                what = "Ignoring the unknown word" if label == "not in our listings" else f"If you are flexible on {label}"
                display(HTML(f"<p><b>{what}</b>: {len(alt):,} car(s) for sale</p>"))
                show_table(alt, limit=8)


def on_search(_):
    run_search(text.value)


def use_example(b):
    text.value = b.description
    run_search(b.description)


button.on_click(on_search)
text.observe(on_search, names="value")          # pressing Enter in the box also searches
example_buttons = []
for e in EXAMPLES:
    b = widgets.Button(description=e, layout=widgets.Layout(width="auto"))
    b.on_click(use_example)
    example_buttons.append(b)

display(HTML("<h3 style='margin-bottom:4px'>Car Finder</h3><div style='color:#64748b;margin-bottom:8px'>"
             "Is it for sale? Describe the car you want, model years 2000–2026.</div>"))
display(widgets.HBox([text, button]))
display(HTML("<div style='color:#64748b;margin-top:8px'>Try:</div>"))
display(widgets.HBox(example_buttons, layout=widgets.Layout(flex_flow="row wrap")))
display(out)

## 4. Limitations
- **The data is synthetic.** Prices, mileage, colours and availability are generated by code, so results show how the system behaves, not the real car market. Upload your own `cars_2000_2026.csv` (same columns) to use real data.
- **Rule-based reading of sentences.** It handles common phrasing but has no spelling correction (`Toyta` counts as an unknown word), no negation (`not red`) and no sorting words (`cheapest`). Descriptive words it does not know, such as `reliable`, are treated as unknown words and can cause a *No*.
- **All conditions must match together.** The closest-alternatives feature drops only one condition at a time.
- **Four-digit numbers can be a year or a price.** `Suzuki Alto 2000` means the year 2000, while `Suzuki Alto under 2000` means a price limit.
- **Prices are in US dollars only,** and English is the only supported language.
- **Not big data.** pandas keeps the whole table in memory, which is perfect for thousands or a few million rows. Much larger data would need a tool such as Spark or a database.
- **Colab sessions are temporary.** The notebook recreates the dataset each time the runtime restarts. Run all cells again after a restart.
- **Testing was done with sample queries,** not with an automated test suite.